In [ ]:
import sys
sys.path.append('/home/lumargot/trachoma/src/py')

import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # put -1 to not use any

In [ ]:
import math
import pandas as pd
import numpy as np 

import torch

import matplotlib.pyplot as plt
import SimpleITK as sitk

In [ ]:
from nets.segmentation import *
from loaders.tt_dataset import TTDataModuleSeg, TrainTransformsSeg, EvalTransformsSeg
from callbacks.logger import SegImageLoggerNeptune, MaskRCNNImageLoggerNeptune


In [ ]:
df_train = pd.read_csv('/CMF/data/lumargot/trachoma/mtss_seg.csv')
df_val = pd.read_csv('/CMF/data/lumargot/trachoma/mtss_seg_train_test.csv')
df_test = pd.read_csv('/CMF/data/lumargot/trachoma/mtss_seg_test.csv')

seg_column='seg'
class_column='class'
img_column='img'
mount_point='/CMF/data/lumargot/trachoma/PoPP_Data/mtss'
drop_labels=['Reject', 'Short Incision']
concat_labels=['overcorrection', 'Gap', 'ECA', 'Fleshy']

In [ ]:
train_transform = TrainTransformsSeg()
eval_transform = EvalTransformsSeg()


ttdata = TTDataModuleSeg(df_train, df_val, df_test, batch_size=1, num_workers=4, img_column=img_column, 
                          seg_column=seg_column, class_column=class_column, mount_point=mount_point, 
                          train_transform=train_transform, valid_transform=eval_transform, test_transform=eval_transform)




In [ ]:
ttdata.setup()
ds = ttdata.test_ds
dl = ttdata.test_dataloader()

In [ ]:
batch = next(iter(dl))

In [ ]:
img,seg =  batch['img'], batch['seg']

In [ ]:
from nets.segmentation import HybridEyelidClassifier

In [ ]:
model = HybridEyelidClassifier(class_weights = [0,0,0])

In [ ]:
out = model.training_step(batch,0)

In [ ]:
from torchvision import models

In [ ]:
effnet = models.efficientnet_b0(pretrained=True)

In [ ]:
conv2d = effnet.features[0]
encod_layer1 = effnet.features[1]
norm_layer1 = nn.InstanceNorm2d(16, affine=True)

encod_layer2 = effnet.features[2]
norm_layer2 = nn.InstanceNorm2d(24, affine=True)

encod_layer3 = effnet.features[3]
norm_layer3 = nn.InstanceNorm2d(40, affine=True)


In [ ]:
from torchvision.transforms import Resize

transform_layer1 = Resize(size=(384, 768))
transform_layer2 = Resize(size=(192, 384))
transform_layer3 = Resize(size=(96, 192))

In [ ]:
list_layers = [ effnet.features[i] for i in range(4,9) ]
decoder = nn.Sequential(*list_layers)

In [ ]:
num_classes =3
fc = nn.Linear(1280, num_classes)

In [ ]:
## encoder block

x = effnet.features[0](img)

x = encod_layer1(x)
downsized_seg = transform_layer1(seg)
x = x + downsized_seg
x = norm_layer1(x)

x = encod_layer2(x)
downsized_seg = transform_layer2(seg)
x = x + downsized_seg
x = norm_layer2(x)

x = encod_layer3(x)
downsized_seg = transform_layer3(seg)
x = x + downsized_seg
x = norm_layer3(x)

x = decoder(x)


In [ ]:
classifier = nn.Sequential(
    nn.Dropout(0.2, inplace=True),
    nn.Linear(1280, num_classes)
)
pooling = nn.AdaptiveAvgPool2d(output_size=1)

In [ ]:
x_pooled = effnet.avgpool(x)
x_pooled = torch.flatten(x_pooled, 1)

In [ ]:
x_pooled.shape

In [ ]:
classifier(x_pooled)

In [ ]:
effnet.classifier(x_pooled)

In [ ]:
instance_norm = nn.InstanceNorm2d(16, affine=True)

In [ ]:
m(x).shape

In [ ]:
plt.imshow(downsized_seg[0,0])
# plt.imshow(img[0].permute(1,2,0))